In [1]:

# Cell 1: Import libraries
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("Libraries loaded successfully.")

Pandas version: 3.0.6
Libraries loaded successfully.


In [2]:

# Cell 2: Load the raw datasets
energy = pd.read_csv("owid-energy-data.csv")
codebook = pd.read_csv("owid-energy-codebook.csv")

print("Energy dataset:", energy.shape)
print("Codebook:", codebook.shape)


Energy dataset: (23377, 130)
Codebook: (130, 5)


In [3]:

# Cell 3: First five rows
energy.head()


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


In [4]:

# Cell 5: Column data types and non-null counts
energy.info()


<class 'pandas.DataFrame'>
RangeIndex: 23377 entries, 0 to 23376
Columns: 130 entries, country to wind_share_energy
dtypes: float64(127), int64(1), str(2)
memory usage: 23.2 MB


In [5]:

# Cell 6: Missing values in the 20 most incomplete columns
energy.isna().mean().sort_values(
    ascending=False
).head(20)


biofuel_cons_change_pct                         0.911451
solar_cons_change_pct                           0.892416
nuclear_cons_change_pct                         0.889678
wind_cons_change_pct                            0.880310
other_renewables_cons_change_pct                0.829405
electricity_share_energy                        0.824486
low_carbon_energy_per_capita                    0.779912
oil_energy_per_capita                           0.779912
coal_cons_per_capita                            0.779912
fossil_energy_per_capita                        0.779912
gas_energy_per_capita                           0.779912
hydro_energy_per_capita                         0.779912
other_renewables_energy_per_capita              0.779912
renewables_energy_per_capita                    0.779912
solar_energy_per_capita                         0.779912
wind_energy_per_capita                          0.779912
other_renewables_elec_per_capita_exc_biofuel    0.769260
gas_cons_change_pct            

In [6]:

# Cell 7: Check duplicate country-year records
energy[["country", "year"]].duplicated().sum()


np.int64(0)

In [7]:

# Cell 8: Select relevant columns
wanted = [
    "country",
    "year",
    "iso_code",
    "population",
    "primary_energy_consumption",
    "electricity_demand",
    "renewables_consumption",
    "fossil_fuel_consumption"
]

wanted = [col for col in wanted if col in energy.columns]

energy_selected = energy[wanted].copy()

energy_selected.head()


,country,year,iso_code,population,primary_energy_consumption,electricity_demand,renewables_consumption,fossil_fuel_consumption
0,ASEAN (Ember),2000,NaN,NaN,NaN,378.76,NaN,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,405.09,NaN,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,433.19,NaN,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,458.30,NaN,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,496.61,NaN,NaN


In [8]:

# Keep rows with a recorded ISO code
country_rows = energy_selected.loc[
    energy_selected["iso_code"].notna()
].copy()

print("Original rows:", len(energy_selected))
print("Rows with ISO codes:", len(country_rows))
print("Rows excluded:", len(energy_selected) - len(country_rows))


Original rows: 23377
Rows with ISO codes: 17265
Rows excluded: 6112


In [9]:

recent = country_rows.loc[
    country_rows["year"].between(2015, 2024)
].copy()

print("Rows from 2015–2024:", len(recent))
print("Earliest year:", recent["year"].min())
print("Latest year:", recent["year"].max())


Rows from 2015–2024: 2179
Earliest year: 2015
Latest year: 2024


In [10]:

recent = recent.sort_values(
    ["country", "year"]
).copy()

recent.head(10)


,country,year,iso_code,population,primary_energy_consumption,electricity_demand,renewables_consumption,fossil_fuel_consumption
141,Afghanistan,2015,AFG,33831767.0,30.954,4.96,NaN,NaN
142,Afghanistan,2016,AFG,34700613.0,28.075,5.54,NaN,NaN
143,Afghanistan,2017,AFG,35688941.0,36.517,5.87,NaN,NaN
144,Afghanistan,2018,AFG,36743040.0,46.493,6.16,NaN,NaN
145,Afghanistan,2019,AFG,37856125.0,47.112,6.37,NaN,NaN
146,Afghanistan,2020,AFG,39068977.0,46.912,6.33,NaN,NaN
147,Afghanistan,2021,AFG,40000410.0,47.569,6.76,NaN,NaN
148,Afghanistan,2022,AFG,40578847.0,45.106,6.84,NaN,NaN
149,Afghanistan,2023,AFG,41454762.0,41.060,7.19,NaN,NaN
150,Afghanistan,2024,AFG,42647502.0,NaN,6.82,NaN,NaN


In [11]:

print("Filtered dataset shape:", recent.shape)

print("\nYear range:")
print(recent["year"].min(), "to", recent["year"].max())

print("\nMissing ISO codes:", recent["iso_code"].isna().sum())

print(
    "Duplicate country-year pairs:",
    recent[["country", "year"]].duplicated().sum()
)


Filtered dataset shape: (2179, 8)

Year range:
2015 to 2024

Missing ISO codes: 0
Duplicate country-year pairs: 0


In [12]:

selected_countries = [
    "Nigeria",
    "Ghana",
    "South Africa",
    "Kenya"
]

comparison = recent.loc[
    recent["country"].isin(selected_countries),
    ["country", "year", "population",
     "primary_energy_consumption",
     "electricity_demand"]
].copy()

comparison.sort_values(
    ["country", "year"]
).tail(20)


,country,year,population,primary_energy_consumption,electricity_demand
14249,Nigeria,2015,190671883.0,468.837,33.17
14250,Nigeria,2016,195443698.0,475.244,34.60
14251,Nigeria,2017,200254580.0,460.025,29.78
14252,Nigeria,2018,204938752.0,491.568,33.71
14253,Nigeria,2019,209485637.0,514.787,34.13
14254,Nigeria,2020,213996185.0,475.403,34.99
14255,Nigeria,2021,218529286.0,512.507,36.14
14256,Nigeria,2022,223150906.0,517.687,35.57
14257,Nigeria,2023,227882949.0,533.847,38.60
14258,Nigeria,2024,232679482.0,NaN,35.26


In [13]:

# Cell 14: Examine missing values
e = recent.copy()

missing_summary = (
    e.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percentage")
    .reset_index()
)

missing_summary.columns = ["column", "missing_percentage"]

missing_summary.head(15)


,column,missing_percentage
0,renewables_consumption,63.744837
1,fossil_fuel_consumption,63.744837
2,primary_energy_consumption,6.333180
3,electricity_demand,3.074805
4,population,0.871960
5,iso_code,0.000000
6,country,0.000000
7,year,0.000000


In [14]:

# Calculate renewable energy share where valid inputs exist
denom = (
    e["renewables_consumption"]
    + e["fossil_fuel_consumption"]
)

e["renewable_share_selected"] = (
    e["renewables_consumption"]
    .div(denom)
    .where(denom.gt(0))
)

# Calculate electricity demand per person where valid
e["electricity_demand_per_capita"] = (
    e["electricity_demand"]
    .div(e["population"])
    .where(e["population"].gt(0))
)

e[
    [
        "country",
        "year",
        "renewables_consumption",
        "fossil_fuel_consumption",
        "renewable_share_selected",
        "electricity_demand_per_capita"
    ]
].head(10)


,country,year,renewables_consumption,fossil_fuel_consumption,renewable_share_selected,electricity_demand_per_capita
141,Afghanistan,2015,NaN,NaN,NaN,1.466078e-07
142,Afghanistan,2016,NaN,NaN,NaN,1.596514e-07
143,Afghanistan,2017,NaN,NaN,NaN,1.644767e-07
144,Afghanistan,2018,NaN,NaN,NaN,1.676508e-07
145,Afghanistan,2019,NaN,NaN,NaN,1.682687e-07
146,Afghanistan,2020,NaN,NaN,NaN,1.620211e-07
147,Afghanistan,2021,NaN,NaN,NaN,1.689983e-07
148,Afghanistan,2022,NaN,NaN,NaN,1.685607e-07
149,Afghanistan,2023,NaN,NaN,NaN,1.734421e-07
150,Afghanistan,2024,NaN,NaN,NaN,1.599156e-07


In [15]:

latest_year = (
    e.groupby("country")["year"]
    .max()
    .rename("latest_year")
)

latest = e.merge(
    latest_year,
    on="country",
    how="left",
    validate="many_to_one"
)

latest = latest.loc[
    latest["year"].eq(latest["latest_year"])
].copy()

latest = latest.drop(columns=["latest_year"])

latest.head()


,country,year,iso_code,population,primary_energy_consumption,electricity_demand,renewables_consumption,fossil_fuel_consumption,renewable_share_selected,electricity_demand_per_capita
9,Afghanistan,2024,AFG,42647502.0,NaN,6.82,NaN,NaN,NaN,1.599156e-07
19,Albania,2024,ALB,2791756.0,NaN,8.67,NaN,NaN,NaN,3.105572e-06
29,Algeria,2024,DZA,46814302.0,760.133,93.84,1.837,758.183,0.002417,2.004516e-06
39,American Samoa,2024,ASM,46792.0,NaN,0.18,NaN,NaN,NaN,3.846811e-06
49,Angola,2024,AGO,37885854.0,NaN,15.59,NaN,NaN,NaN,4.114992e-07


In [16]:

period_summary = (
    e.groupby("country")
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        observations=("year", "count"),
        mean_primary_energy=("primary_energy_consumption", "mean"),
        mean_electricity_demand=("electricity_demand", "mean"),
        mean_renewable_share=("renewable_share_selected", "mean")
    )
    .reset_index()
)

period_summary.head(10)


,country,first_year,last_year,observations,mean_primary_energy,mean_electricity_demand,mean_renewable_share
0,Afghanistan,2015,2024,10,41.088667,6.284,NaN
1,Albania,2015,2024,10,25.337222,7.835,NaN
2,Algeria,2015,2024,10,685.771700,81.340,0.002225
3,American Samoa,2015,2024,10,1.483333,0.168,NaN
4,Angola,2015,2024,10,92.593111,14.284,NaN
5,Antarctica,2015,2023,9,0.053556,NaN,NaN
6,Antigua and Barbuda,2015,2024,10,3.001556,0.355,NaN
7,Argentina,2015,2024,10,972.518100,156.278,0.118840
8,Armenia,2015,2024,10,42.548444,6.886,NaN
9,Aruba,2015,2024,10,4.896444,0.983,NaN


In [17]:

value_cols = [
    col for col in [
        "primary_energy_consumption",
        "electricity_demand"
    ]
    if col in e.columns
]

long = e.melt(
    id_vars=["country", "iso_code", "year"],
    value_vars=value_cols,
    var_name="column",
    value_name="value"
)

long.head(10)


,country,iso_code,year,column,value
0,Afghanistan,AFG,2015,primary_energy_consumption,30.954
1,Afghanistan,AFG,2016,primary_energy_consumption,28.075
2,Afghanistan,AFG,2017,primary_energy_consumption,36.517
3,Afghanistan,AFG,2018,primary_energy_consumption,46.493
4,Afghanistan,AFG,2019,primary_energy_consumption,47.112
5,Afghanistan,AFG,2020,primary_energy_consumption,46.912
6,Afghanistan,AFG,2021,primary_energy_consumption,47.569
7,Afghanistan,AFG,2022,primary_energy_consumption,45.106
8,Afghanistan,AFG,2023,primary_energy_consumption,41.060
9,Afghanistan,AFG,2024,primary_energy_consumption,NaN


In [18]:

cb = codebook.drop_duplicates("column").copy()

long_meta = long.merge(
    cb,
    on="column",
    how="left",
    validate="many_to_one",
    indicator=True
)

print(long_meta["_merge"].value_counts())

long_meta.head(10)


_merge
both          4358
left_only        0
right_only       0
Name: count, dtype: int64


,country,iso_code,year,column,value,title,description,unit,source,_merge
0,Afghanistan,AFG,2015,primary_energy_consumption,30.954,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
1,Afghanistan,AFG,2016,primary_energy_consumption,28.075,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
2,Afghanistan,AFG,2017,primary_energy_consumption,36.517,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
3,Afghanistan,AFG,2018,primary_energy_consumption,46.493,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
4,Afghanistan,AFG,2019,primary_energy_consumption,47.112,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
5,Afghanistan,AFG,2020,primary_energy_consumption,46.912,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
6,Afghanistan,AFG,2021,primary_energy_consumption,47.569,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
7,Afghanistan,AFG,2022,primary_energy_consumption,45.106,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
8,Afghanistan,AFG,2023,primary_energy_consumption,41.060,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both
9,Afghanistan,AFG,2024,primary_energy_consumption,NaN,Primary energy consumption,Measured in terawatt-hours.,terawatt-hours (TWh),U.S. Energy Information Administration - Inter...,both


In [19]:

# Validation 1: No missing country or year identifiers
assert e["country"].notna().all()
assert e["year"].notna().all()

# Validation 2: No duplicate country-year pairs
assert not e[["country", "year"]].duplicated().any()

# Validation 3: All retained years are in the requested period
assert e["year"].between(2015, 2024).all()

print("All three validation checks passed.")


All three validation checks passed.


In [20]:

# Check that the merge preserved the expected number of rows
assert len(long_meta) == len(long)

# Check that each measurement has a matching codebook entry
unmatched = long_meta.loc[
    long_meta["_merge"] != "both",
    "column"
].unique()

print("Rows before merge:", len(long))
print("Rows after merge:", len(long_meta))
print("Unmatched indicators:", unmatched)


Rows before merge: 4358
Rows after merge: 4358
Unmatched indicators: <StringArray>
[]
Length: 0, dtype: str


In [21]:

health_report = pd.DataFrame({
    "check": [
        "Number of rows",
        "Number of columns",
        "Unique countries",
        "Earliest year",
        "Latest year",
        "Missing country values",
        "Missing year values",
        "Duplicate country-year pairs"
    ],
    "result": [
        len(e),
        len(e.columns),
        e["country"].nunique(),
        e["year"].min(),
        e["year"].max(),
        e["country"].isna().sum(),
        e["year"].isna().sum(),
        e[["country", "year"]].duplicated().sum()
    ]
})

health_report


,check,result
0,Number of rows,2179
1,Number of columns,10
2,Unique countries,220
3,Earliest year,2015
4,Latest year,2024
5,Missing country values,0
6,Missing year values,0
7,Duplicate country-year pairs,0


In [22]:

from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output folder:", output_dir.resolve())


Output folder: C:\Users\kingj\Documents\Data-Science\data-acqusition\outputs


In [23]:

# Export the main processed dataset
e.to_csv(output_dir / "energy_cleaned.csv", index=False)

# Export the latest available observations
latest.to_csv(output_dir / "latest_by_country.csv", index=False)

# Export the country-level summary
period_summary.to_csv(
    output_dir / "country_period_summary.csv",
    index=False
)

# Export the reshaped measurements with codebook metadata
long_meta.drop(columns=["_merge"]).to_csv(
    output_dir / "energy_with_codebook.csv",
    index=False
)

# Export data quality reports
missing_summary.to_csv(
    output_dir / "missing_values_report.csv",
    index=False
)

health_report.to_csv(
    output_dir / "dataset_health_report.csv",
    index=False
)

print("All six CSV exports completed.")


All six CSV exports completed.


In [24]:

# Calculate report figures from the processed data
row_count = len(e)
country_count = e["country"].nunique()
first_year = int(e["year"].min())
last_year = int(e["year"].max())
duplicate_count = int(
    e[["country", "year"]].duplicated().sum()
)
missing_country = int(e["country"].isna().sum())
missing_year = int(e["year"].isna().sum())
matched_rows = int((long_meta["_merge"] == "both").sum())
unmatched_rows = int((long_meta["_merge"] != "both").sum())

management_note = f"""
MANAGEMENT NOTE: OUR WORLD IN DATA ENERGY ANALYSIS

Objective
This analysis examines country-level energy indicators over the
2015–2024 period to support structured comparison of energy
consumption and electricity demand.

Dataset and methodology
The analysis used the supplied Our World in Data energy CSV and
its indicator codebook. The working dataset contains {row_count}
records representing {country_count} distinct country labels.
The observed year range is {first_year}–{last_year}. Records without
an ISO code were excluded during the initial country-selection
step. Country-year records were sorted and checked for duplicates.

Data preparation
Missing values were quantified rather than automatically replaced
with zero, because unavailable measurements do not necessarily
represent zero consumption. Two features were calculated where
valid inputs were available: renewable consumption divided by
combined renewable and fossil-fuel consumption, and electricity
demand per person. These derived indicators have different
denominators and should not be interpreted interchangeably.

Analysis and validation
Country-level summaries were produced for observation counts,
period coverage, mean primary energy consumption, mean electricity
demand and mean derived renewable share. A separate table records
the latest available observation for each country. Selected energy
indicators were reshaped into long format and merged with the
codebook. The merge matched {matched_rows} measurements and left
{unmatched_rows} measurements unmatched. Duplicate country-year
pairs: {duplicate_count}. Missing country identifiers: {missing_country};
missing year identifiers: {missing_year}.

Limitations and recommendations
Data availability may differ across countries and years, so averages
can be based on different numbers of observations. The ISO-code
filter is a useful initial selection rule but does not by itself
prove that every retained entity is a sovereign country. The derived
renewable share excludes energy sources outside its stated
denominator. Results should therefore be interpreted alongside
the missingness report, indicator definitions and coverage dates.
Before policy decisions, verify entity classifications, investigate
unmatched codebook indicators and compare countries over consistent
periods wherever possible.
"""

print("Word count:", len(management_note.split()))
print(management_note)


Word count: 297

MANAGEMENT NOTE: OUR WORLD IN DATA ENERGY ANALYSIS

Objective
This analysis examines country-level energy indicators over the
2015–2024 period to support structured comparison of energy
consumption and electricity demand.

Dataset and methodology
The analysis used the supplied Our World in Data energy CSV and
its indicator codebook. The working dataset contains 2179
records representing 220 distinct country labels.
The observed year range is 2015–2024. Records without
an ISO code were excluded during the initial country-selection
step. Country-year records were sorted and checked for duplicates.

Data preparation
Missing values were quantified rather than automatically replaced
with zero, because unavailable measurements do not necessarily
represent zero consumption. Two features were calculated where
valid inputs were available: renewable consumption divided by
combined renewable and fossil-fuel consumption, and electricity
demand per person. These derived indicators 

In [25]:

(output_dir / "management_note.txt").write_text(
    management_note.strip(),
    encoding="utf-8"
)

exported_files = sorted(
    file.name for file in output_dir.iterdir()
)

print("Files saved:")
for filename in exported_files:
    print("-", filename)


Files saved:
- country_period_summary.csv
- dataset_health_report.csv
- energy_cleaned.csv
- energy_with_codebook.csv
- latest_by_country.csv
- management_note.txt
- missing_values_report.csv
